In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

from transformers import AutoTokenizer, AutoModel

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path


LANGUAGE_PAIRS = [
    "km-en",
    "th-en",
    "vi-en",
    "my-en",
]


def read_json_or_jsonl(path):
    """
    Hỗ trợ:
    1. JSON array: [{...}, {...}]
    2. JSON object chứa trường data/items
    3. JSONL: mỗi dòng là một JSON object
    """
    with open(path, "r", encoding="utf-8") as file:
        content = file.read().strip()

    if not content:
        raise ValueError(f"File rỗng: {path}")

    # Thử đọc như một JSON hoàn chỉnh trước.
    try:
        data = json.loads(content)

        if isinstance(data, list):
            return data

        if isinstance(data, dict):
            for key in ("data", "items", "records"):
                if isinstance(data.get(key), list):
                    return data[key]

            # Một file chỉ chứa một mẫu
            return [data]

    except json.JSONDecodeError:
        pass

    # Nếu không phải JSON hoàn chỉnh, đọc theo JSONL.
    records = []

    for line_number, line in enumerate(content.splitlines(), start=1):
        line = line.strip()

        if not line:
            continue

        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as error:
            raise ValueError(
                f"JSON không hợp lệ tại {path}, "
                f"dòng {line_number}: {error}"
            ) from error

    return records


def extract_parallel_text(record, lang1, lang2, path, row_index):
    """
    Trích câu song song từ trường:
        "translation": {
            "km": "...",
            "en": "..."
        }
    """
    translation = record.get("translation")

    if not isinstance(translation, dict):
        raise ValueError(
            f"{path}, mẫu {row_index}: không có dictionary 'translation'. "
            f"Các trường hiện có: {list(record.keys())}"
        )

    missing_languages = [
        lang for lang in (lang1, lang2)
        if lang not in translation
    ]

    if missing_languages:
        raise ValueError(
            f"{path}, mẫu {row_index}: thiếu ngôn ngữ "
            f"{missing_languages} trong translation. "
            f"Các ngôn ngữ hiện có: {list(translation.keys())}"
        )

    text1 = translation[lang1]
    text2 = translation[lang2]

    if text1 is None or text2 is None:
        raise ValueError(
            f"{path}, mẫu {row_index}: có câu translation bị null."
        )

    text1 = str(text1).strip()
    text2 = str(text2).strip()

    if not text1 or not text2:
        raise ValueError(
            f"{path}, mẫu {row_index}: có câu translation rỗng."
        )

    return text1, text2


def load_parallel_mt50_json(
    data_dir,
    language_pairs=LANGUAGE_PAIRS,
    max_samples=None,
):
    data_dir = Path(data_dir)

    frames = []
    lengths = {}
    selected_files = {}

    for pair in language_pairs:
        pair_dir = data_dir / pair

        if not pair_dir.exists():
            raise FileNotFoundError(
                f"Không tìm thấy thư mục: {pair_dir}"
            )

        # Ví dụ: test.km-en.general_trans.wmt23.json
        test_paths = sorted(pair_dir.glob("test.*.json"))

        if not test_paths:
            raise FileNotFoundError(
                f"Không tìm thấy file test.*.json trong: {pair_dir}"
            )

        if len(test_paths) > 1:
            raise ValueError(
                f"Có nhiều file test trong {pair_dir}: "
                f"{[p.name for p in test_paths]}. "
                "Hãy chỉ định rõ file cần dùng."
            )

        path = test_paths[0]
        selected_files[pair] = str(path)

        lang1, lang2 = pair.split("-")
        records = read_json_or_jsonl(path)

        if max_samples is not None:
            records = records[:max_samples]

        pair_rows = {
            lang1: [],
            lang2: [],
        }

        for row_index, record in enumerate(records):
            if not isinstance(record, dict):
                raise ValueError(
                    f"{path}, mẫu {row_index} không phải JSON object."
                )

            text1, text2 = extract_parallel_text(
                record=record,
                lang1=lang1,
                lang2=lang2,
                path=path,
                row_index=row_index,
            )

            pair_rows[lang1].append(text1)
            pair_rows[lang2].append(text2)

        lengths[pair] = len(records)

        for language in (lang1, lang2):
            texts = pair_rows[language]

            frame = pd.DataFrame({
                "text": texts,
                "sentence_id": np.arange(
                    len(texts),
                    dtype=np.int64,
                ),
                "language": language,
                "language_pair": pair,
                "source_file": path.name,
            })

            frames.append(frame)

    df = pd.concat(frames, ignore_index=True)

    df = df.sort_values(
        ["language_pair", "language", "sentence_id"]
    ).reset_index(drop=True)

    return df, lengths, selected_files

In [ ]:
DATA_DIR = "/kaggle/input/datasets/truonghai/mt-50k"
MAX_SAMPLES_PER_LANG = None

df, pair_lengths, selected_files = load_parallel_mt50_json(
    data_dir=DATA_DIR,
    language_pairs=["km-en", "th-en", "vi-en", "my-en"],
    max_samples=MAX_SAMPLES_PER_LANG,
)

print("Các file đã đọc:")
for pair, path in selected_files.items():
    print(f"{pair}: {path}")

print("\nSố cặp câu:")
print(pair_lengths)

print("\nTổng số câu sau khi chuyển sang dạng long:", len(df))

print("\nSố câu theo cặp và ngôn ngữ:")
print(
    df.groupby(["language_pair", "language"])
      .size()
      .rename("count")
)

display(df.head())

In [ ]:
def masked_mean_pooling(hidden_state, attention_mask):
    # hidden_state: [B, T, D]
    # attention_mask: [B, T]
    mask = attention_mask.unsqueeze(-1).to(hidden_state.dtype)

    summed = (hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)

    return summed / counts


In [ ]:
BATCH_SIZE = 32
MAX_LENGTH = 256

tokenizer.padding_side = "left"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


@torch.inference_mode()
def extract_all_layer_embeddings(
    texts,
    tokenizer,
    model,
    batch_size=8,
    max_length=256
):
    all_layers = None

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc="Extracting hidden states"
    ):
        batch_texts = texts[start:start + batch_size]

        encoded = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        )

        # Với device_map='auto', inputs nên chuyển vào device
        # của embedding layer/model đầu vào.
        input_device = next(model.parameters()).device
        encoded = {
            k: v.to(input_device)
            for k, v in encoded.items()
        }

        outputs = model(
            **encoded,
            output_hidden_states=True,
            return_dict=True,
            use_cache=False
        )

        hidden_states = outputs.hidden_states

        # hidden_states[0]: token embedding trước transformer blocks
        # hidden_states[i]: output sau transformer layer i
        if all_layers is None:
            all_layers = [
                [] for _ in range(len(hidden_states))
            ]

        for layer_idx, hidden in enumerate(hidden_states):
            pooled = masked_mean_pooling(
                hidden,
                encoded["attention_mask"]
            )

            pooled = pooled.float().cpu().numpy()
            all_layers[layer_idx].append(pooled)

        del outputs, hidden_states, encoded
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return [
        np.concatenate(layer_batches, axis=0)
        for layer_batches in all_layers
    ]


texts = df["text"].tolist()

layer_embeddings = extract_all_layer_embeddings(
    texts=texts,
    tokenizer=tokenizer,
    model=model,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH
)

print("Number of hidden-state tensors:", len(layer_embeddings))
print("Layer 0 shape:", layer_embeddings[0].shape)
print("Final layer shape:", layer_embeddings[-1].shape)


In [ ]:
import torch
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.manifold import TSNE
from matplotlib.lines import Line2D


def plot_multilingual_tsne_scatter(
    embeddings_dict,
    sample_size=None,
    layers=None,
    perplexity=30,
    cols=3,
    figsize_scale=4,
    random_state=42,
    point_size=14,
    alpha=0.8,
):
    """
    Vẽ t-SNE scatter cho embedding đa ngôn ngữ,
    kèm nền KDE của toàn bộ phân phối.

    embeddings_dict:
        {
            "en": Tensor[num_layers, num_samples, hidden_dim],
            "vi": Tensor[num_layers, num_samples, hidden_dim],
            "km": Tensor[num_layers, num_samples, hidden_dim],
            ...
        }
    """

    if not embeddings_dict:
        raise ValueError("embeddings_dict đang rỗng.")

    labels = list(embeddings_dict.keys())
    first_tensor = next(iter(embeddings_dict.values()))

    if not torch.is_tensor(first_tensor) or first_tensor.ndim != 3:
        raise ValueError(
            "Embedding phải có shape "
            "[num_layers, num_samples, hidden_dim]."
        )

    num_layers = first_tensor.shape[0]
    hidden_dim = first_tensor.shape[2]

    for name, emb in embeddings_dict.items():
        if not torch.is_tensor(emb):
            raise TypeError(
                f"{name}: embedding phải là torch.Tensor."
            )

        if emb.ndim != 3:
            raise ValueError(
                f"{name}: nhận được shape {tuple(emb.shape)}, "
                "expected tensor 3D."
            )

        if emb.shape[0] != num_layers:
            raise ValueError(
                f"{name}: số layer không nhất quán."
            )

        if emb.shape[2] != hidden_dim:
            raise ValueError(
                f"{name}: hidden dimension không nhất quán."
            )

    if layers is None:
        layers = list(range(7, num_layers))
    else:
        layers = list(layers)

    invalid_layers = [
        layer
        for layer in layers
        if not 0 <= layer < num_layers
    ]

    if invalid_layers:
        raise ValueError(
            f"Layer không hợp lệ: {invalid_layers}."
        )

    if not layers:
        raise ValueError("Không có layer nào để visualize.")

    min_num_samples = min(
        emb.shape[1]
        for emb in embeddings_dict.values()
    )

    if sample_size is None:
        sample_size = min_num_samples
    else:
        if sample_size <= 0:
            raise ValueError("sample_size phải lớn hơn 0.")

        sample_size = min(
            sample_size,
            min_num_samples
        )

    total_samples = sample_size * len(labels)

    if total_samples < 2:
        raise ValueError(
            "Cần ít nhất 2 embedding để chạy t-SNE."
        )

    effective_perplexity = min(
        perplexity,
        total_samples - 1
    )

    # Mỗi ngôn ngữ có một màu riêng biệt
    language_colors = {
    "en": "#1f77b4",  # xanh dương
    "km": "#ff7f0e",  # cam
    "my": "#2ca02c",  # xanh lá
    "th": "#d62728",  # đỏ
    "vi": "#9467bd",  # tím
}

    palette = {
        label: language_colors[label]
        for label in labels
    }

    # palette = dict(zip(labels, colors))

    rows = (len(layers) + cols - 1) // cols

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(
            figsize_scale * cols,
            figsize_scale * rows
        ),
        squeeze=False,
    )

    axes = axes.flatten()

    legend_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="",
            markerfacecolor=palette[name],
            markeredgecolor="white",
            markeredgewidth=0.3,
            markersize=6,
            label=name,
        )
        for name in labels
    ]

    for plot_idx, layer in enumerate(layers):
        ax = axes[plot_idx]

        embedding_groups = []

        for name in labels:
            layer_embedding = embeddings_dict[name][
                layer, :sample_size
            ]

            embedding_groups.append(layer_embedding)

        # Chạy chung t-SNE cho mọi ngôn ngữ
        X = (
            torch.cat(embedding_groups, dim=0)
            .detach()
            .float()
            .cpu()
            .numpy()
        )

        tsne = TSNE(
            n_components=2,
            perplexity=effective_perplexity,
            init="pca",
            learning_rate="auto",
            random_state=random_state,
        )

        X2 = tsne.fit_transform(X)

        # Nền KDE xám của toàn bộ phân phối
        if X2.shape[0] >= 5:
            sns.kdeplot(
                x=X2[:, 0],
                y=X2[:, 1],
                fill=True,
                levels=12,
                thresh=0.03,
                bw_adjust=0.8,
                color="#BDBDBD",
                alpha=0.38,
                ax=ax,
                zorder=1,
            )

        # Scatter từng ngôn ngữ
        start = 0

        for name in labels:
            end = start + sample_size
            points = X2[start:end]

            ax.scatter(
                points[:, 0],
                points[:, 1],
                s=point_size,
                color=palette[name],
                alpha=alpha,
                edgecolors="white",
                linewidths=0.2,
                zorder=2,
            )

            start = end

        ax.set_title(
            f"Layer {layer}",
            fontsize=12
        )

        ax.set_xlabel("x")
        ax.set_ylabel("y")

        ax.grid(
            True,
            linestyle="-",
            linewidth=0.4,
            alpha=0.25,
        )

        ax.legend(
            handles=legend_handles,
            loc="upper right",
            frameon=True,
            framealpha=0.9,
            fontsize=8,
            ncol=1 if len(labels) <= 6 else 2,
        )

    for idx in range(len(layers), len(axes)):
        axes[idx].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
import torch


def build_embeddings_dict(
    layer_embeddings,
    df,
    language_column="language",
    sentence_id_column="sentence_id",
    source_language="en",
):
    """
    Convert:
        list[layer] -> [N_total, hidden_dim]

    thành:
        dict[lang] -> [num_layers, N_lang, hidden_dim]

    Các câu thuộc source_language sẽ được loại trùng
    theo sentence_id.
    """

    # Đảm bảo vị trí hàng khớp với layer_embeddings
    working_df = df.reset_index(drop=True).copy()
    working_df["_embedding_row"] = range(len(working_df))

    embeddings_dict = {}

    languages = sorted(
        working_df[language_column].unique()
    )

    for lang in languages:
        lang_df = working_df[
            working_df[language_column] == lang
        ].copy()

        # Tiếng Anh xuất hiện lặp lại trong các file song ngữ
        if lang == source_language:
            lang_df = lang_df.drop_duplicates(
                subset=[sentence_id_column],
                keep="first",
            )

        lang_df = lang_df.sort_values(
            sentence_id_column
        )

        row_positions = (
            lang_df["_embedding_row"]
            .to_numpy()
        )

        layer_tensors = []

        for emb in layer_embeddings:
            selected = emb[row_positions]

            if torch.is_tensor(selected):
                x = (
                    selected
                    .detach()
                    .float()
                    .cpu()
                )
            else:
                x = torch.as_tensor(
                    selected,
                    dtype=torch.float32,
                )

            layer_tensors.append(x)

        embeddings_dict[lang] = torch.stack(
            layer_tensors,
            dim=0,
        )

    return embeddings_dict

In [ ]:
embeddings_dict = build_embeddings_dict(
    layer_embeddings,
    df,
    source_language="en",
)

for lang, emb in embeddings_dict.items():
    print(lang, tuple(emb.shape))

In [ ]:
plot_multilingual_tsne_scatter(
    embeddings_dict,
    sample_size=1000,
    layers=[2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 28],
    # layers=[8],
    perplexity=30,
    cols=3,
)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.trainer_utils import get_last_checkpoint
from peft import PeftModel
import torch

MODEL_ID = "/kaggle/working/model-stage1-ot"
TRAIN_OUTPUT_DIR = "/kaggle/working//Multilingual/outputs/stage2-sft"

# Lấy checkpoint cuối
LORA_PATH = get_last_checkpoint(TRAIN_OUTPUT_DIR)

if LORA_PATH is None:
    raise ValueError(f"Không tìm thấy checkpoint trong {TRAIN_OUTPUT_DIR}")

print("Load LoRA from:", LORA_PATH)

# Base model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

# LoRA adapter
model = PeftModel.from_pretrained(
    model,
    LORA_PATH,
    is_trainable=False,
)

model = model.merge_and_unload()

model.eval()

In [ ]:
embeddings_dict = build_embeddings_dict(
    layer_embeddings,
    df,
    source_language="en",
)

for lang, emb in embeddings_dict.items():
    print(lang, tuple(emb.shape))

In [ ]:
plot_multilingual_tsne_scatter(
    embeddings_dict,
    sample_size=1000,
    layers=[2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 28],
    # layers=[8],
    perplexity=30,
    cols=3,
)